# Topic and hybrid filtering

This notebook stores fresh predictions in a corpus, defines a topic filter, and combines it with a regex filter. Edit the topic identifiers and thresholds only after inspecting the fitted model and score distribution.

In [ ]:
%env PS_DB=papers.db
%env PS_MODEL_DIR=topic_model
%env PS_MODEL_NAME=domain-lda-v1

## Store current predictions

Storage predicts afresh and records the source-text fingerprint. This prevents an old external CSV from silently becoming an active filter.

In [ ]:
%%bash
set -euo pipefail
ps_topics_store "$PS_MODEL_DIR" "$PS_DB" --name "$PS_MODEL_NAME" --batch-size 1000
ps_topics_models "$PS_DB"

## Create reviewable filter definitions

The example includes papers scoring at least 0.35 on topic 2 and vetoes papers dominated by topic 7. Replace both identifiers after manual topic naming.

In [ ]:
import json
from pathlib import Path

topic_filter = {
    "name": "focused-domain-topics",
    "description": "Manually reviewed LDA relevance rule",
    "model": "domain-lda-v1",
    "include_mode": "any",
    "include": [{"name": "target-topic", "topic_id": 2, "min_probability": 0.35, "require_dominant": False}],
    "exclude": [{"name": "unrelated-topic", "topic_id": 7, "require_dominant": True}],
}
regex_filter = {
    "name": "domain-keywords",
    "fields": ["title", "abstract"],
    "case_sensitive": False,
    "include_mode": "any",
    "timeout_ms": 500,
    "include": [{"name": "target-phrase", "pattern": r"\bsolid[ -]?electrolytes?\b"}],
    "exclude": [],
}
Path("topic_filter.json").write_text(json.dumps(topic_filter, indent=2) + "\n", encoding="utf-8")
Path("regex_filter.json").write_text(json.dumps(regex_filter, indent=2) + "\n", encoding="utf-8")

## Apply topic-only or hybrid filtering

For a topic-only test, apply only the first command. The complete sequence requires both keyword relevance and the selected topic rule because filters are evaluated left to right.

In [ ]:
%%bash
set -euo pipefail
ps_filter_reset "$PS_DB" --all
ps_filter_regex "$PS_DB" regex_filter.json
ps_filter_topic "$PS_DB" topic_filter.json --join and
ps_filter_status "$PS_DB"

## Use and maintain the filter

`ps_scrape` now processes only the final included set. If corpus text changes, the stored model is marked stale and scraping fails closed. Refresh with `ps_topics_store`, then reapply or replace the filter definition. Clear experiments with `ps_filter_reset papers.db --all`.